# 04 — Split, formatação e publicação

**Entrada:** `../data/corpus_reescrito.parquet` (saída do `03`).
**Saída:** `../data/{train,val,test}.jsonl`, amostra em `../dataset_exemplo/` e dataset no HuggingFace.

Notebook barato até a última célula (o push). Pode ser re-executado à vontade.

In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"
CORPUS_REESCRITO = DATA / "corpus_reescrito.parquet"
SEED = 42

final = pd.read_parquet(CORPUS_REESCRITO)
print(f"{len(final):,} linhas | {final['condition'].nunique()} condições "
      f"| {final['cluster_id'].nunique():,} clusters")
final.head(3)

15,300 linhas | 616 condições | 10,195 clusters


,id,condition,medical_specialty,question_type,cluster_id,cluster_size,tam_q,tam_a,origem,fonte,question,answer,question_original,answer_original,acao,pergunta_reescrita,resposta_reescrita
0,1612,Bruxismo,Pediatra,Diagnóstico,1471,1,160,61,medpt,AKCIT/MedPT,Paciente pediátrico com bruxismo noturno prévi...,"Não, não é normal. Indica-se avaliação com ped...",Meu filho apresentou bruxismo noturno e após u...,"Não, não é normal, o leve para avaliação com p...",reescrever_ambos,True,True
1,1613,Bruxismo,Pediatra,Diagnóstico,1472,1,160,708,medpt,AKCIT/MedPT,Criança com bruxismo noturno que evoluiu para ...,O bruxismo diurno pode estar relacionado a mud...,Meu filho apresentou bruxismo noturno e após u...,É possível que o bruxismo diurno esteja relaci...,reescrever_ambos,True,True
2,1701,Bruxismo,Pediatra,Escolha de profissionais de saúde,1473,1,104,94,medpt,AKCIT/MedPT,Diante de um lactente de 6 meses com bruxismo ...,"Sim, a primeira consulta com dentista deve ser...",Meu bebê tem 6 meses tem 4 dentes ele fica rig...,"Sim, a primeira consulta com dentista deve ser...",reescrever_ambos,True,True


## Split por grupo, não aleatório

O `02` preserva duplicatas de propósito — frequência de pergunta é sinal sobre o que
médicos de fato perguntam. Mas isso cria um risco: com near-duplicatas no corpus, um
split aleatório coloca a mesma pergunta (em variação) no treino **e** no teste, e a
métrica passa a medir memorização.

A solução é fechar o problema por construção, não por corte posterior:
**`cluster_id` vira o grupo do split**. `StratifiedGroupKFold` garante que um cluster
inteiro cai num único split, ao mesmo tempo em que equilibra a distribuição de condições
entre eles. Isso libera manter redundância no treino sem risco nenhum de vazamento.

Substitui o `train_test_split` estratificado da versão anterior, que não tinha noção de
grupo e espalhava near-duplicatas entre os splits.

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

MIN_ESTRATO = 15   # condicoes abaixo disso entram no estrato "raras"
N_FOLDS = 10       # 10 folds -> teste 10%, val 10%, treino 80%

cont = final["condition"].value_counts()
estrato = final["condition"].where(final["condition"].map(cont) >= MIN_ESTRATO, "raras")
print(f"{estrato.nunique()} estratos ({(estrato == 'raras').sum():,} linhas em 'raras')")

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = [teste for _, teste in sgkf.split(final, estrato, groups=final["cluster_id"])]

idx_teste = folds[0]
idx_val = folds[1]
idx_treino = np.concatenate(folds[2:])

splits = {
    "train": final.iloc[idx_treino].copy(),
    "val": final.iloc[idx_val].copy(),
    "test": final.iloc[idx_teste].copy(),
}
for nome, df in splits.items():
    print(f"{nome:<6} {len(df):>7,} linhas  {len(df)/len(final):>5.1%}")

118 estratos (1,665 linhas em 'raras')
train   12,240 linhas  80.0%
val      1,530 linhas  10.0%
test     1,530 linhas  10.0%


In [12]:
# Verificacao dura: nenhum cluster pode aparecer em mais de um split.
dono = {}
vazamentos = []
for nome, df in splits.items():
    for cid in df["cluster_id"].unique():
        if cid in dono:
            vazamentos.append((cid, dono[cid], nome))
        else:
            dono[cid] = nome

assert not vazamentos, f"VAZAMENTO: {len(vazamentos)} clusters em mais de um split"
print(f"OK — {len(dono):,} clusters, cada um em exatamente um split")

soma = sum(len(d) for d in splits.values())
assert soma == len(final), f"perdeu linhas: {soma} != {len(final)}"
print(f"OK — {soma:,} linhas conservadas")

OK — 10,195 clusters, cada um em exatamente um split
OK — 15,300 linhas conservadas


In [13]:
# Cobertura das obstetricas criticas no teste (o gate do item #8).
RARAS_OBST = ["Hemorragia Pós-Parto", "Trabalho De Parto Prematuro",
              "Abortamento Incompleto", "Ameaça De Abortamento", "Gravidez Tubária"]

print(f"{'condição':<32} {'treino':>7} {'val':>5} {'teste':>6}")
for cond in RARAS_OBST:
    n = {k: int((d["condition"] == cond).sum()) for k, d in splits.items()}
    marca = "  <-- ausente do teste" if n["test"] == 0 else ""
    print(f"{cond:<32} {n['train']:>7} {n['val']:>5} {n['test']:>6}{marca}")

ausentes = [c for c in RARAS_OBST
            if (splits['test']['condition'] == c).sum() == 0
            and (final['condition'] == c).sum() > 0]
if ausentes:
    print(f"\n{len(ausentes)} condição(ões) crítica(s) sem representação no teste.")
    print("Não avaliável nessas condições — ver item #8 do plano antes de sintetizar.")

condição                          treino   val  teste
Hemorragia Pós-Parto                   6     0      1
Trabalho De Parto Prematuro            6     0      0  <-- ausente do teste
Abortamento Incompleto                12     0      0  <-- ausente do teste
Ameaça De Abortamento                  4     3      0  <-- ausente do teste
Gravidez Tubária                       5     3      2

3 condição(ões) crítica(s) sem representação no teste.
Não avaliável nessas condições — ver item #8 do plano antes de sintetizar.


## Formato de treino

Formato `messages`, que é o que os frameworks de fine-tuning consomem hoje.

O `SYSTEM_ASSISTENTE` carrega o **limite de atuação** exigido pela fase: nunca prescrever
sem validação humana. Colocá-lo em toda linha de treino faz o modelo aprender a fronteira
como parte do comportamento, em vez de depender só do prompt em tempo de inferência.

Os campos de metadado (`id`, `origem`, `fonte`, `cluster_id`) viajam junto para
rastreabilidade e para a explicabilidade das respostas. `pergunta_reescrita` e
`resposta_reescrita` vêm do `03`: o segundo marca as linhas cujo alvo de treino é saída de
LLM em vez de texto de médico, e é o que permite avaliar o modelo separando as duas
populações depois.

In [14]:
SYSTEM_ASSISTENTE = (
    "Voce e um assistente clinico de um hospital maternidade, que apoia profissionais "
    "de saude no acompanhamento de gestantes, puerperas e bebes ate 1 ano.\n\n"
    "Voce responde a medicos, em registro tecnico. Voce apoia a decisao clinica; voce "
    "nao a substitui. Nunca prescreva medicamento, dose ou conduta como determinacao "
    "final: toda sugestao precisa de validacao do profissional responsavel. "
    "Quando a informacao disponivel nao sustentar uma resposta, diga isso em vez de "
    "preencher a lacuna."
)

def para_messages(row):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_ASSISTENTE},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "id": int(row["id"]),
        "condition": row["condition"],
        "question_type": row["question_type"],
        "cluster_id": int(row["cluster_id"]),
        "origem": row["origem"],
        "fonte": row["fonte"],
        "pergunta_reescrita": bool(row["pergunta_reescrita"]),
        "resposta_reescrita": bool(row["resposta_reescrita"]),
    }

for nome, df in splits.items():
    destino = DATA / f"{nome}.jsonl"
    with destino.open("w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(para_messages(row), ensure_ascii=False) + "\n")
    print(f"{destino.name:<12} {len(df):>7,} linhas  "
          f"({1 - df['resposta_reescrita'].mean():.0%} com alvo humano)")

print(json.dumps(para_messages(splits["train"].iloc[0]), ensure_ascii=False, indent=2)[:700])

train.jsonl   12,240 linhas  (15% com alvo humano)
val.jsonl      1,530 linhas  (16% com alvo humano)
test.jsonl     1,530 linhas  (16% com alvo humano)
{
  "messages": [
    {
      "role": "system",
      "content": "Voce e um assistente clinico de um hospital maternidade, que apoia profissionais de saude no acompanhamento de gestantes, puerperas e bebes ate 1 ano.\n\nVoce responde a medicos, em registro tecnico. Voce apoia a decisao clinica; voce nao a substitui. Nunca prescreva medicamento, dose ou conduta como determinacao final: toda sugestao precisa de validacao do profissional responsavel. Quando a informacao disponivel nao sustentar uma resposta, diga isso em vez de preencher a lacuna."
    },
    {
      "role": "user",
      "content": "Lactente de 11 meses, com 4 dentes superiores e 4 inferiores, apresenta bruxismo intermitente, 


## Amostra para o repositório

A fase pede um dataset anonimizado ou exemplo de dados sintéticos **no repositório Git**.
O `data/` está no `.gitignore` inteiro, e abrir exceção dentro de diretório ignorado não
funciona no git — negação sob diretório excluído é inerte, porque o git nem desce nele.

Por isso a amostra sai em `dataset_exemplo/`, no topo do projeto: pasta versionada,
sem precisar de padrão de exceção.

In [15]:
AMOSTRA_POR_SPLIT = 50
EXEMPLO = RAIZ / "dataset_exemplo"
EXEMPLO.mkdir(exist_ok=True)

for nome, df in splits.items():
    amostra = df.sample(min(AMOSTRA_POR_SPLIT, len(df)), random_state=SEED)
    destino = EXEMPLO / f"{nome}_amostra.jsonl"
    with destino.open("w", encoding="utf-8") as f:
        for _, row in amostra.iterrows():
            f.write(json.dumps(para_messages(row), ensure_ascii=False) + "\n")
    print(f"{destino.name:<24} {len(amostra):>4} linhas")

train_amostra.jsonl        50 linhas
val_amostra.jsonl          50 linhas
test_amostra.jsonl         50 linhas


## Publicação no HuggingFace

Duas ressalvas antes de rodar:

- **Licença.** O card do MedPT não declara licença — nem nas tags nem no metadata. Ausência
  de licença não é permissão: não há concessão explícita para redistribuir derivados.
  Enquanto isso não estiver resolvido, `PRIVADO = True`. A citação do paper está no card e
  deve constar no README do dataset publicado de qualquer forma.
- **Conteúdo reescrito.** Parte das respostas é saída de LLM, não texto de médico. O card
  do dataset publicado precisa dizer isso, com o número — que sai da última célula — e
  apontar `resposta_reescrita` como o campo que separa as duas populações linha a linha.

In [16]:
REPO_ID = "emidiosouza/maternidade-qa"   # ajuste
PRIVADO = False
PUBLICAR = True                          # vire para True quando decidir publicar

if PUBLICAR:
    from datasets import Dataset, DatasetDict

    ds = DatasetDict({
        nome: Dataset.from_pandas(
            pd.DataFrame([para_messages(r) for _, r in df.iterrows()]),
            preserve_index=False,
        )
        for nome, df in splits.items()
    })
    print(ds)
    ds.push_to_hub(REPO_ID, private=PRIVADO)
    print(f"publicado em {REPO_ID} (privado={PRIVADO})")
else:
    print("PUBLICAR=False — nada enviado.")

DatasetDict({
    train: Dataset({
        features: ['messages', 'id', 'condition', 'question_type', 'cluster_id', 'origem', 'fonte', 'pergunta_reescrita', 'resposta_reescrita'],
        num_rows: 12240
    })
    val: Dataset({
        features: ['messages', 'id', 'condition', 'question_type', 'cluster_id', 'origem', 'fonte', 'pergunta_reescrita', 'resposta_reescrita'],
        num_rows: 1530
    })
    test: Dataset({
        features: ['messages', 'id', 'condition', 'question_type', 'cluster_id', 'origem', 'fonte', 'pergunta_reescrita', 'resposta_reescrita'],
        num_rows: 1530
    })
})


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 41.11ba/s]
Processing Files (1 / 1): 100%|██████████| 13.6MB / 13.6MB, 1.17MB/s  
New Data Upload: 100%|██████████| 13.6MB / 13.6MB, 1.17MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.94s/ shards]
Setting num_proc from 1 back to 1 for the val split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 227.58ba/s]
Processing Files (1 / 1): 100%|██████████| 1.72MB / 1.72MB,  155kB/s  
New Data Upload: 100%|██████████| 1.72MB / 1.72MB,  155kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.05s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 291

publicado em emidiosouza/maternidade-qa (privado=False)


## Resumo para o relatório

Os números que a seção de dados do relatório técnico precisa.

In [18]:
print(f"{'':16} {'linhas':>8} {'condições':>10} {'clusters':>9} {'P reescr':>9} {'R reescr':>9}")
for nome, df in splits.items():
    print(f"{nome:<16} {len(df):>8,} {df['condition'].nunique():>10} "
          f"{df['cluster_id'].nunique():>9,} {df['pergunta_reescrita'].mean():>8.0%} "
          f"{df['resposta_reescrita'].mean():>8.0%}")
print(f"{'TOTAL':<16} {len(final):>8,} {final['condition'].nunique():>10} "
      f"{final['cluster_id'].nunique():>9,} {final['pergunta_reescrita'].mean():>8.0%} "
      f"{final['resposta_reescrita'].mean():>8.0%}")

print(f"\nprocedência: {final['origem'].value_counts().to_dict()}")
print(f"tipos de pergunta: {final['question_type'].value_counts().to_dict()}")
print(f"tamanho de cluster: mediana {final['cluster_size'].median():.0f}, "
      f"máx {final['cluster_size'].max():.0f}")

# A frase que vai para a secao de limitacoes do relatorio.
destilado = final["resposta_reescrita"].mean()
print(f"\n{1 - destilado:.1%} das respostas do dataset são texto de médico preservado "
      f"byte a byte.")
print(f"{destilado:.1%} foram reescritas por LLM (glm-5.3-flash) e devem ser "
      f"tratadas como destilação.")

                   linhas  condições  clusters  P reescr  R reescr
train              12,240        574     8,157     100%      85%
val                 1,530        238     1,019     100%      84%
test                1,530        240     1,019     100%      84%
TOTAL              15,300        616    10,195     100%      85%

procedência: {'medpt': 15300}
tipos de pergunta: {'Diagnóstico': 5825, 'Tratamento': 4955, 'Epidemiologia': 2849, 'Estilo de vida saudável': 733, 'Outros': 420, 'Anatomia e fisiologia': 279, 'Escolha de profissionais de saúde': 239}
tamanho de cluster: mediana 2, máx 25

15.4% das respostas do dataset são texto de médico preservado byte a byte.
84.6% foram reescritas por LLM (glm-5.3-flash) e devem ser tratadas como destilação.
